# 01 — Dataset Audit: Northern Bangladesh Heart Disease Dataset

**Source:** `Resource files/Heart_diasease_dataset_from_Northern_Bangladesh.xlsx`
(sheet `Our Dataset`). The original file is read-only here and never modified.

This notebook recomputes the audit directly from the raw Excel file — it is
the reproducible companion to `reports/dataset_audit.md` and
`reports/feature_groups.md`. All outputs below are real, executed results,
not illustrative examples.

This is a fresh audit pass for a new Basic/Enhanced/Advanced feature-tiering
architecture. It is independent of, and cross-checked against, the project's
earlier executed notebook
(`Resource files/Northern_Bangladesh_Heart_Disease_ML_Revised.executed.ipynb`),
which trained a single-tier deployed model. Nothing here modifies that
earlier notebook or the model it produced.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

DATA_PATH = r"../Resource files/Heart_diasease_dataset_from_Northern_Bangladesh.xlsx"

xl = pd.ExcelFile(DATA_PATH)
print("Sheets:", xl.sheet_names)
df_raw = pd.read_excel(DATA_PATH, sheet_name=xl.sheet_names[0])
print("Shape:", df_raw.shape)
df_raw.columns.tolist()

Sheets: ['Our Dataset']


Shape: (1048, 27)


['SL',
 ' Age',
 'Sex',
 'Height (cm)',
 'Weight (kg)',
 'BMI',
 'Family H/O',
 'Hypertension',
 'Diabetes',
 'Total_Cholesterol(mg/dL)',
 'BP(mmHg)',
 'H/O ChestPain',
 'RBS(mmol/L)',
 'HDL(mg/dL)',
 'LDL(mg/dL)',
 'Triglycerides(mg/dL)',
 'MaxHR',
 'Himoglobin',
 'Creatinine(mg/dL)',
 'Platelets',
 'Sodium(mmol/L)',
 'Potassium',
 'Chloride',
 'Troponin-I ',
 'Troponin- I assay type',
 'Heart Disease',
 'UNIT']

## 1. Column Name Cleanup

Three column names carry stray whitespace (`' Age'`, `'Troponin-I '`,
`'Troponin- I assay type'`) which silently breaks direct name lookups on the
raw file. Stripped here before any further analysis; no values are changed.

In [2]:
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]
df.columns.tolist()

['SL',
 'Age',
 'Sex',
 'Height (cm)',
 'Weight (kg)',
 'BMI',
 'Family H/O',
 'Hypertension',
 'Diabetes',
 'Total_Cholesterol(mg/dL)',
 'BP(mmHg)',
 'H/O ChestPain',
 'RBS(mmol/L)',
 'HDL(mg/dL)',
 'LDL(mg/dL)',
 'Triglycerides(mg/dL)',
 'MaxHR',
 'Himoglobin',
 'Creatinine(mg/dL)',
 'Platelets',
 'Sodium(mmol/L)',
 'Potassium',
 'Chloride',
 'Troponin-I',
 'Troponin- I assay type',
 'Heart Disease',
 'UNIT']

## 2. Structural Overview: dtypes, missingness, uniqueness

In [3]:
overview = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_n': df.isna().sum(),
    'missing_pct': (df.isna().sum() / len(df) * 100).round(2),
    'n_unique': df.nunique(dropna=True),
})
overview.sort_values('missing_pct', ascending=False)

,dtype,missing_n,missing_pct,n_unique
Hypertension,float64,99,9.45,2
BMI,float64,91,8.68,561
Chloride,object,78,7.44,208
Platelets,float64,74,7.06,251
Potassium,object,73,6.97,65
Sodium(mmol/L),float64,71,6.77,240
Family H/O,float64,68,6.49,2
Height (cm),float64,66,6.30,46
Himoglobin,object,51,4.87,121
Triglycerides(mg/dL),float64,51,4.87,234


In [4]:
print("Exact duplicate rows:", df.duplicated().sum())
print("SL is literally row position (SL == index+1) for all rows:", (df['SL'] == (df.index + 1)).all())

Exact duplicate rows: 0
SL is literally row position (SL == index+1) for all rows: True


## 3. Target Distribution

In [5]:
target_counts = df['Heart Disease'].value_counts()
target_pct = df['Heart Disease'].value_counts(normalize=True).round(4) * 100
pd.DataFrame({'n': target_counts, 'pct': target_pct})

,n,pct
Heart Disease,,
1,598,57.06
0,450,42.94


## 4. Pediatric Records (Age < 18)

All-positive pediatric records would make "is this patient a child" a
mechanical, perfect predictor if left in the modelling population — a
population-definition issue, not a feature-engineering one.

In [6]:
pediatric = df[df['Age'] < 18]
print("Pediatric record count:", len(pediatric))
print("Ages:", sorted(pediatric['Age'].tolist()))
print("Heart Disease value counts among pediatric records:")
print(pediatric['Heart Disease'].value_counts())

Pediatric record count: 13
Ages: [4, 4, 5, 6, 6, 8, 9, 11, 11, 12, 13, 15, 16]
Heart Disease value counts among pediatric records:
Heart Disease
1    13
Name: count, dtype: int64


## 5. Text-Stored 'Numeric' Columns — Stray Value Detection

`Himoglobin`, `Potassium`, `Chloride`, and `Troponin-I` are stored as text
(`object` dtype) despite being semantically numeric. Each has a small number
of non-numeric-parseable string artifacts on top of genuine values —
detected explicitly here rather than silently coerced (which would turn
censored troponin values and formatting artifacts into indistinguishable
`NaN`s).

In [7]:
def is_clean_float(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

for c in ['Himoglobin', 'Potassium', 'Chloride', 'Troponin-I']:
    vals = df[c].dropna().astype(str)
    bad = vals[~vals.apply(is_clean_float)]
    print(f"{c}: {len(bad)} non-numeric-parseable value(s) -> {bad.unique().tolist()}")

Himoglobin: 1 non-numeric-parseable value(s) -> ['`2.6']
Potassium: 1 non-numeric-parseable value(s) -> ['4,3']
Chloride: 2 non-numeric-parseable value(s) -> ['106,4', '106. 8']
Troponin-I: 21 non-numeric-parseable value(s) -> ['>25000', '<2.50', '>2.5']


## 6. Troponin-I — Assay Types, Censoring, Range

Two assay types on incompatible scales are mixed in one column, and
censoring appears on **both** ends (not just the high end previously
documented) — `<2.50` is a genuine low-censored value alongside the
`>25000`-style high-censored ones.

In [8]:
def try_float(s):
    try:
        return float(s)
    except ValueError:
        return np.nan

troponin_str = df['Troponin-I'].astype(str)
censored_mask = troponin_str.str.contains('>', na=False) | troponin_str.str.contains('<', na=False)
print("Censored value count:", censored_mask.sum())
print("Censored values:", df.loc[censored_mask, 'Troponin-I'].unique().tolist())

df['troponin_numeric'] = df['Troponin-I'].apply(try_float)
print()
print("Per-assay-type range (parseable values only):")
print(df.groupby('Troponin- I assay type')['troponin_numeric'].agg(['count', 'min', 'median', 'max']))

Censored value count: 21
Censored values: ['>25000', '<2.50', '>2.5']

Per-assay-type range (parseable values only):
                                    count    min   median       max
Troponin- I assay type                                             
High-Sensitivity Troponin-I (ng/L)    181  45.00  1075.61  65860.00
Quantitative Troponin-I (ng/mL)       800   0.01     1.14     33.95


## 7. UNIT vs Target — Leakage Confirmation

Admission ward is a downstream/concurrent clinical decision, not information
available at the point a real prediction would be made.

In [9]:
unit_crosstab = pd.crosstab(df['UNIT'], df['Heart Disease'])
unit_crosstab['positive_rate'] = (unit_crosstab[1] / unit_crosstab.sum(axis=1)).round(3)
unit_crosstab

Heart Disease,0,1,positive_rate
UNIT,,,
CCU,0,370,1.000
General,450,228,0.336


## 8. Correlation with Target

Computed on numeric/binary columns (text-stored numeric columns parsed via
`pd.to_numeric(errors='coerce')` for this correlation check only).

In [10]:
numeric_binary_cols = [
    'Age', 'Height (cm)', 'Weight (kg)', 'BMI', 'Family H/O', 'Hypertension',
    'Diabetes', 'Total_Cholesterol(mg/dL)', 'BP(mmHg)', 'H/O ChestPain',
    'RBS(mmol/L)', 'HDL(mg/dL)', 'LDL(mg/dL)', 'Triglycerides(mg/dL)', 'MaxHR',
    'Creatinine(mg/dL)', 'Platelets', 'Sodium(mmol/L)',
]
corrs = {c: df[c].corr(df['Heart Disease']) for c in numeric_binary_cols}
for c in ['Himoglobin', 'Potassium', 'Chloride']:
    corrs[c] = pd.to_numeric(df[c], errors='coerce').corr(df['Heart Disease'])
corrs['Troponin-I'] = df['troponin_numeric'].corr(df['Heart Disease'])

corr_series = pd.Series(corrs).sort_values(key=abs, ascending=False)
corr_series.round(3)

LDL(mg/dL)                  0.674
Total_Cholesterol(mg/dL)    0.601
Triglycerides(mg/dL)        0.566
Himoglobin                 -0.524
H/O ChestPain               0.449
RBS(mmol/L)                 0.443
Age                         0.427
Creatinine(mg/dL)           0.374
MaxHR                      -0.373
Diabetes                    0.346
Family H/O                  0.333
Sodium(mmol/L)             -0.271
Hypertension                0.271
Weight (kg)                 0.259
BMI                         0.250
Troponin-I                  0.226
HDL(mg/dL)                 -0.181
Chloride                    0.103
BP(mmHg)                   -0.094
Platelets                  -0.072
Height (cm)                 0.025
Potassium                  -0.024
dtype: float64

**Continuity flag:** the four strongest correlates — **LDL, Total
Cholesterol, Triglycerides, Haemoglobin** — are exactly the same four the
earlier notebook flagged as *abnormally high* for a retrospective clinical
cohort and treated as an unresolved limitation (label-construction question
never independently verifiable — see project memory / `dataset_audit.md`
§8). Reproducing them independently here from the raw file is a useful
confirmation, not a resolution — the same sensitivity-analysis caution
carries forward into the new experiments.

## 9. Derived-Variable Check: MaxHR vs Age

In [11]:
print("Correlation(MaxHR, Age):", df['MaxHR'].corr(df['Age']))

Correlation(MaxHR, Age): -0.9258478053361537


Strongly negative, consistent with the earlier notebook's finding that
MaxHR is substantially explained by an age-based formula (R² ≈ 0.85 for
MaxHR ≈ 206.13 − 0.75×Age). This is why MaxHR is placed in the Enhanced tier
rather than Basic in `feature_groups.md` despite its individual correlation
with the outcome — it's largely redundant with Age, which is already Basic.

## 10. Implausible-Value Screening (loose sanity bounds, not clinical thresholds)

In [12]:
sanity_bounds = {
    'Age': (0, 120), 'Height (cm)': (100, 220), 'Weight (kg)': (20, 200),
    'BMI': (10, 60), 'BP(mmHg)': (60, 250), 'MaxHR': (60, 220),
    'RBS(mmol/L)': (2, 40), 'Total_Cholesterol(mg/dL)': (80, 500),
    'HDL(mg/dL)': (10, 120), 'LDL(mg/dL)': (20, 350),
    'Triglycerides(mg/dL)': (30, 800), 'Creatinine(mg/dL)': (0.2, 15),
    'Platelets': (20000, 700000), 'Sodium(mmol/L)': (110, 170),
}
any_flagged = False
for col, (lo, hi) in sanity_bounds.items():
    s = pd.to_numeric(df[col], errors='coerce')
    out = s[(s < lo) | (s > hi)]
    if len(out):
        any_flagged = True
        print(f"{col}: {len(out)} value(s) outside [{lo}, {hi}] -> {sorted(out.tolist())}")
if not any_flagged:
    print("No values outside the loose sanity bounds checked above.")
print()
print("The 3 low-weight values (17, 17, 18 kg) belong to the pediatric records (see section 4) — plausible, not errors.")

Weight (kg): 3 value(s) outside [20, 200] -> [17.0, 17.0, 18.0]



The 3 low-weight values (17, 17, 18 kg) belong to the pediatric records (see section 4) — plausible, not errors.


## 11. Summary

See `../reports/dataset_audit.md` for the full written audit and
`../reports/feature_groups.md` for the resulting Basic / Enhanced / Advanced
/ Excluded feature architecture and open questions awaiting approval before
Phase ML-3 (preprocessing) begins.